In [39]:
import warnings
warnings.simplefilter(action='ignore',category=FutureWarning)


from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import (
    train_test_split,
    cross_val_score
) 
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv("Telco_churn_data.xls")
df.head()
df.isnull().sum()
df.describe()
df.nunique()
df.head()
yes_no_columns = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
    "Churn"
]
df[yes_no_columns]= df[yes_no_columns].replace({
    "Yes":1,
    "No":0
})
df["gender"] = df["gender"].replace({
    "Male":1,
    "Female":0
})
df.drop(columns="customerID",inplace = True)
df = pd.get_dummies(
    df,
    columns = [
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "PaymentMethod",
        "Contract"
    ],
    drop_first=True
)
pd.to_numeric(df["TotalCharges"],errors='coerce')
mask = pd.to_numeric(df["TotalCharges"],errors='coerce').isna()
df = df[~mask]
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"])
#line below is meant to address skewness
df["TotalCharges"] = np.log1p(df["TotalCharges"]) 
cols_to_scale = ["tenure","MonthlyCharges","TotalCharges"]
scalr = StandardScaler()
df[cols_to_scale] = scalr.fit_transform(df[cols_to_scale])
df["TotalCharges"].head()
df = df.drop(
    columns=[
        'OnlineSecurity_No internet service',
        'DeviceProtection_No internet service',
        'StreamingTV_No internet service',
        'OnlineBackup_No internet service',
        'TechSupport_No internet service',
        'StreamingMovies_No internet service'
    ])

In [72]:
X = df.drop(columns=["Churn","gender"],axis=1)
Y=df["Churn"]
X_train,X_test,Y_train,Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)
model = LogisticRegression(class_weight='balanced', C= 0.01)
model.fit(X_train,Y_train)
prediction = model.predict(X_test)
accuracy = accuracy_score(Y_test,prediction)
cm = confusion_matrix(Y_test,prediction)
cr = classification_report(Y_test,prediction)
print(accuracy)
print(cm)
print(cr)

0.7420042643923241
[[752 281]
 [ 82 292]]
              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1033
           1       0.51      0.78      0.62       374

    accuracy                           0.74      1407
   macro avg       0.71      0.75      0.71      1407
weighted avg       0.80      0.74      0.76      1407



In [65]:
probs = model.predict_proba(X_test)[:,1]
for threshold in [0.3,0.45,0.5,0.6,0.7]:
    pred = (probs>threshold).astype(int)
    pro_cm = confusion_matrix(Y_test,pred)
    pro_cr = classification_report(Y_test,pred)
    print(pro_cm)
    print(pro_cr)


[[585 448]
 [ 33 341]]
              precision    recall  f1-score   support

           0       0.95      0.57      0.71      1033
           1       0.43      0.91      0.59       374

    accuracy                           0.66      1407
   macro avg       0.69      0.74      0.65      1407
weighted avg       0.81      0.66      0.68      1407

[[694 339]
 [ 58 316]]
              precision    recall  f1-score   support

           0       0.92      0.67      0.78      1033
           1       0.48      0.84      0.61       374

    accuracy                           0.72      1407
   macro avg       0.70      0.76      0.70      1407
weighted avg       0.81      0.72      0.73      1407

[[725 308]
 [ 72 302]]
              precision    recall  f1-score   support

           0       0.91      0.70      0.79      1033
           1       0.50      0.81      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weigh

In [33]:
df['OnlineSecurity_No internet service'].value_counts()
df['StreamingMovies_No internet service'].value_counts()
df['StreamingTV_No internet service'].value_counts()

StreamingTV_No internet service
False    5512
True     1520
Name: count, dtype: int64

In [73]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 25 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 7032 non-null   int64  
 1   SeniorCitizen                          7032 non-null   int64  
 2   Partner                                7032 non-null   int64  
 3   Dependents                             7032 non-null   int64  
 4   tenure                                 7032 non-null   float64
 5   PhoneService                           7032 non-null   int64  
 6   PaperlessBilling                       7032 non-null   int64  
 7   MonthlyCharges                         7032 non-null   float64
 8   TotalCharges                           7032 non-null   float64
 9   Churn                                  7032 non-null   int64  
 10  MultipleLines_No phone service         7032 non-null   bool   
 11  MultipleL

In [76]:
corr = df.corr()['Churn'].sort_values(ascending=True)
print(corr)

tenure                                  -0.354049
Contract_Two year                       -0.301552
TotalCharges                            -0.241908
InternetService_No                      -0.227578
Contract_One year                       -0.178225
OnlineSecurity_Yes                      -0.171270
TechSupport_Yes                         -0.164716
Dependents                              -0.163128
Partner                                 -0.149982
PaymentMethod_Credit card (automatic)   -0.134687
PaymentMethod_Mailed check              -0.090773
OnlineBackup_Yes                        -0.082307
DeviceProtection_Yes                    -0.066193
MultipleLines_No phone service          -0.011691
gender                                  -0.008545
PhoneService                             0.011691
MultipleLines_Yes                        0.040033
StreamingMovies_Yes                      0.060860
StreamingTV_Yes                          0.063254
SeniorCitizen                            0.150541


In [80]:
print(df['tenure'].corr(df['Contract_Two year']))

0.5638005002286525
